In [ ]:
# ASSIGNMENT 5 – CO5
from pathlib import Path
import subprocess
import zipfile
import shutil
import os
import urllib.request

PROJECT = Path("/content/CSA09-Assignment5")
PROJECT.mkdir(parents=True, exist_ok=True)

if shutil.which("javac") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "openjdk-17-jdk-headless", "-qq"],
        check=True
    )


In [ ]:
%%writefile /content/CSA09-Assignment5/Task01_FileStore/PassStore.java
import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.io.IOException;
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.StandardOpenOption;
import java.util.ArrayList;
import java.util.List;

public class PassStore {
    private final Path file;

    public PassStore(Path file) {
        this.file = file;
    }

    public void add(String passId, String regNo, String name, int route, double amount)
            throws IOException {
        try (BufferedWriter writer = Files.newBufferedWriter(
                file, StandardOpenOption.CREATE, StandardOpenOption.APPEND)) {
            writer.write(String.join(",",
                    clean(passId), clean(regNo), clean(name),
                    String.valueOf(route), String.valueOf(amount)));
            writer.newLine();
        }
    }

    private String clean(String value) {
        return value == null ? "" : value.replace(",", " ").replace("\n", " ").trim();
    }

    public String[] find(String passId) throws IOException {
        if (!Files.exists(file)) return null;
        try (BufferedReader reader = Files.newBufferedReader(file)) {
            String line;
            while ((line = reader.readLine()) != null) {
                String[] fields = line.split(",", -1);
                if (fields.length != 5) continue;
                try {
                    Integer.parseInt(fields[3].trim());
                    Double.parseDouble(fields[4].trim());
                } catch (NumberFormatException damaged) {
                    continue;
                }
                if (fields[0].trim().equals(passId)) return fields;
            }
        }
        return null;
    }

    public List<String[]> listByRoute(int route) throws IOException {
        List<String[]> matches = new ArrayList<>();
        if (!Files.exists(file)) return matches;
        try (BufferedReader reader = Files.newBufferedReader(file)) {
            String line;
            while ((line = reader.readLine()) != null) {
                String[] fields = line.split(",", -1);
                if (fields.length != 5) continue;
                try {
                    if (Integer.parseInt(fields[3].trim()) == route)
                        matches.add(fields);
                    Double.parseDouble(fields[4].trim());
                } catch (NumberFormatException ignored) {
                    // Ignore damaged records and continue scanning.
                }
            }
        }
        return matches;
    }

    public static void main(String[] args) throws Exception {
        Path path = Files.createTempFile("bus-passes", ".csv");
        PassStore store = new PassStore(path);
        store.add("P1001", "192311001", "Arun", 12, 4500);
        Files.writeString(path, "P1002,DAMAGED LINE\n", StandardOpenOption.APPEND);
        store.add("P1003", "192311003", "Divya", 7, 3800);
        System.out.println(java.util.Arrays.toString(store.find("P1003")));
        System.out.println(java.util.Arrays.toString(store.find("P1002")));
        System.out.println("Route 7 count: " + store.listByRoute(7).size());
        Files.deleteIfExists(path);
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task01_FileStore/BufferedLineCounter.java
import java.io.BufferedInputStream;
import java.io.FileInputStream;
import java.io.IOException;
import java.nio.file.Files;
import java.nio.file.Path;

public class BufferedLineCounter {
    public static long countUnbuffered(Path file) throws IOException {
        long lines = 0;
        boolean hasData = false;
        try (FileInputStream in = new FileInputStream(file.toFile())) {
            int value;
            while ((value = in.read()) != -1) {
                hasData = true;
                if (value == '\n') lines++;
            }
        }
        return lines + (hasData && !endsWithNewline(file) ? 1 : 0);
    }

    public static long countBuffered(Path file) throws IOException {
        long lines = 0;
        boolean hasData = false;
        try (BufferedInputStream in =
                     new BufferedInputStream(new FileInputStream(file.toFile()), 64 * 1024)) {
            int value;
            while ((value = in.read()) != -1) {
                hasData = true;
                if (value == '\n') lines++;
            }
        }
        return lines + (hasData && !endsWithNewline(file) ? 1 : 0);
    }

    private static boolean endsWithNewline(Path file) throws IOException {
        long size = Files.size(file);
        if (size == 0) return true;
        try (var channel = java.nio.channels.FileChannel.open(file)) {
            java.nio.ByteBuffer one = java.nio.ByteBuffer.allocate(1);
            channel.position(size - 1);
            channel.read(one);
            return one.array()[0] == '\n';
        }
    }

    public static void main(String[] args) throws Exception {
        Path file = Files.createTempFile("attendance", ".log");
        StringBuilder data = new StringBuilder();
        for (int i = 0; i < 100000; i++) data.append("ATTENDANCE-").append(i).append('\n');
        Files.writeString(file, data);
        long start = System.nanoTime();
        long a = countUnbuffered(file);
        long unbufferedMs = (System.nanoTime() - start) / 1_000_000;
        start = System.nanoTime();
        long b = countBuffered(file);
        long bufferedMs = (System.nanoTime() - start) / 1_000_000;
        System.out.println("Lines: " + a + " / " + b);
        System.out.println("Unbuffered ms: " + unbufferedMs);
        System.out.println("Buffered ms: " + bufferedMs);
        Files.deleteIfExists(file);
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task02_DataAccess/Dao.java
import java.sql.SQLException;
import java.util.List;

public interface Dao<T, K> {
    int insert(T item) throws SQLException;
    T findById(K id) throws SQLException;
    List<T> findAll() throws SQLException;
    int update(T item) throws SQLException;
    int delete(K id) throws SQLException;
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task02_DataAccess/Student.java
public class Student {
    private final int studentId;
    private final String registerNumber;
    private final String name;
    private final String email;

    public Student(int studentId, String registerNumber, String name, String email) {
        this.studentId = studentId;
        this.registerNumber = registerNumber;
        this.name = name;
        this.email = email;
    }

    public int getStudentId() { return studentId; }
    public String getRegisterNumber() { return registerNumber; }
    public String getName() { return name; }
    public String getEmail() { return email; }

    @Override
    public String toString() {
        return studentId + " | " + registerNumber + " | " + name + " | " + email;
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task02_DataAccess/StudentDao.java
import java.sql.*;
import java.util.ArrayList;
import java.util.List;

public class StudentDao implements Dao<Student, Integer> {
    private final Connection connection;

    public StudentDao(Connection connection) {
        this.connection = connection;
    }

    @Override
    public int insert(Student s) throws SQLException {
        String sql = "INSERT INTO student(student_id, reg_no, name, email) VALUES (?, ?, ?, ?)";
        try (PreparedStatement ps = connection.prepareStatement(sql)) {
            ps.setInt(1, s.getStudentId());
            ps.setString(2, s.getRegisterNumber());
            ps.setString(3, s.getName());
            ps.setString(4, s.getEmail());
            return ps.executeUpdate();
        }
    }

    @Override
    public Student findById(Integer id) throws SQLException {
        String sql = "SELECT student_id, reg_no, name, email FROM student WHERE student_id = ?";
        try (PreparedStatement ps = connection.prepareStatement(sql)) {
            ps.setInt(1, id);
            try (ResultSet rs = ps.executeQuery()) {
                return rs.next() ? new Student(rs.getInt("student_id"),
                        rs.getString("reg_no"), rs.getString("name"), rs.getString("email")) : null;
            }
        }
    }

    @Override
    public List<Student> findAll() throws SQLException {
        List<Student> students = new ArrayList<>();
        try (Statement st = connection.createStatement();
             ResultSet rs = st.executeQuery(
                     "SELECT student_id, reg_no, name, email FROM student ORDER BY student_id")) {
            while (rs.next()) students.add(new Student(rs.getInt(1), rs.getString(2),
                    rs.getString(3), rs.getString(4)));
        }
        return students;
    }

    @Override
    public int update(Student s) throws SQLException {
        String sql = "UPDATE student SET reg_no=?, name=?, email=? WHERE student_id=?";
        try (PreparedStatement ps = connection.prepareStatement(sql)) {
            ps.setString(1, s.getRegisterNumber());
            ps.setString(2, s.getName());
            ps.setString(3, s.getEmail());
            ps.setInt(4, s.getStudentId());
            return ps.executeUpdate();
        }
    }

    @Override
    public int delete(Integer id) throws SQLException {
        try (PreparedStatement ps =
                     connection.prepareStatement("DELETE FROM student WHERE student_id=?")) {
            ps.setInt(1, id);
            return ps.executeUpdate();
        }
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task03_Serialization/Hostel.java
import java.io.Serializable;
import java.util.ArrayList;
import java.util.List;

public class Hostel implements Serializable {
    private static final long serialVersionUID = 1L;
    private final String hostelName;
    private final List<Room> rooms = new ArrayList<>();

    public Hostel(String hostelName) { this.hostelName = hostelName; }
    public void addRoom(Room room) { rooms.add(room); }
    public String getHostelName() { return hostelName; }
    public List<Room> getRooms() { return rooms; }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task03_Serialization/Room.java
import java.io.Serializable;
import java.util.ArrayList;
import java.util.List;

public class Room implements Serializable {
    private static final long serialVersionUID = 1L;
    private final int roomNumber;
    private final List<Resident> residents = new ArrayList<>();

    public Room(int roomNumber) { this.roomNumber = roomNumber; }
    public void addResident(Resident resident) { residents.add(resident); }
    public int getRoomNumber() { return roomNumber; }
    public List<Resident> getResidents() { return residents; }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task03_Serialization/Resident.java
import java.io.Serializable;
import java.sql.Connection;

public class Resident implements Serializable {
    private static final long serialVersionUID = 1L;
    private final String name;
    private final String registerNumber;
    private final String phoneNumber;
    private transient String password;
    private transient Connection liveConnection;

    public Resident(String name, String registerNumber, String phoneNumber,
                    String password, Connection liveConnection) {
        this.name = name;
        this.registerNumber = registerNumber;
        this.phoneNumber = phoneNumber;
        this.password = password;
        this.liveConnection = liveConnection;
    }

    public String getName() { return name; }
    public String getRegisterNumber() { return registerNumber; }
    public String getPhoneNumber() { return phoneNumber; }
    public String getPassword() { return password; }
    public Connection getLiveConnection() { return liveConnection; }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task03_Serialization/SerializationDemo.java
import java.io.*;

public class SerializationDemo {
    public static void main(String[] args) throws Exception {
        Hostel hostel = new Hostel("SIMATS Hostel");
        Room room = new Room(101);
        room.addResident(new Resident("Arun", "192311001", "9876543210",
                "not-persisted", null));
        hostel.addRoom(room);

        File file = new File("hostel-state.ser");
        try (ObjectOutputStream out = new ObjectOutputStream(new FileOutputStream(file))) {
            out.writeObject(hostel);
        }
        try (ObjectInputStream in = new ObjectInputStream(new FileInputStream(file))) {
            Hostel restored = (Hostel) in.readObject();
            Resident resident = restored.getRooms().get(0).getResidents().get(0);
            System.out.println(restored.getHostelName());
            System.out.println(resident.getName() + " " + resident.getPhoneNumber());
            System.out.println("Password after load: " + resident.getPassword());
            System.out.println("Connection after load: " + resident.getLiveConnection());
        }
        file.delete();
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task04_Transactions/WalletTransfer.java
import java.math.BigDecimal;
import java.sql.Connection;
import java.sql.PreparedStatement;
import java.sql.SQLException;

public class WalletTransfer {
    public static void transfer(Connection con, int fromWallet, int toWallet,
                                 BigDecimal amount) throws SQLException {
        if (amount == null || amount.signum() <= 0)
            throw new IllegalArgumentException("Amount must be positive");

        boolean oldAutoCommit = con.getAutoCommit();
        con.setAutoCommit(false);
        try {
            try (PreparedStatement debit = con.prepareStatement(
                    "UPDATE wallet SET balance = balance - ? WHERE wallet_id = ? AND balance >= ?");
                 PreparedStatement credit = con.prepareStatement(
                    "UPDATE wallet SET balance = balance + ? WHERE wallet_id = ?")) {
                debit.setBigDecimal(1, amount);
                debit.setInt(2, fromWallet);
                debit.setBigDecimal(3, amount);
                if (debit.executeUpdate() != 1)
                    throw new SQLException("Debit failed: wallet missing or insufficient balance");

                credit.setBigDecimal(1, amount);
                credit.setInt(2, toWallet);
                if (credit.executeUpdate() != 1)
                    throw new SQLException("Credit failed: destination wallet not found");
            }
            con.commit();
        } catch (SQLException | RuntimeException ex) {
            con.rollback();
            throw ex;
        } finally {
            con.setAutoCommit(oldAutoCommit);
        }
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task04_Transactions/LoginSecurityDemo.java
import java.sql.*;

public class LoginSecurityDemo {
    public static boolean unsafeLogin(Connection con, String id, String password)
            throws SQLException {
        String sql = "SELECT COUNT(*) FROM app_user WHERE user_id='" + id
                + "' AND user_pwd='" + password + "'";
        try (Statement st = con.createStatement(); ResultSet rs = st.executeQuery(sql)) {
            return rs.next() && rs.getInt(1) > 0;
        }
    }

    public static boolean safeLogin(Connection con, String id, String password)
            throws SQLException {
        String sql = "SELECT COUNT(*) FROM app_user WHERE user_id=? AND user_pwd=?";
        try (PreparedStatement ps = con.prepareStatement(sql)) {
            ps.setString(1, id);
            ps.setString(2, password);
            try (ResultSet rs = ps.executeQuery()) {
                return rs.next() && rs.getInt(1) > 0;
            }
        }
    }

    public static void main(String[] args) throws Exception {
        String url = System.getenv().getOrDefault("DB_URL",
                "jdbc:mysql://localhost:3306/assignment5");
        String user = System.getenv().getOrDefault("DB_USER", "root");
        String password = System.getenv().getOrDefault("DB_PASSWORD", "");
        try (Connection con = DriverManager.getConnection(url, user, password)) {
            String attack = "' OR '1'='1";
            System.out.println("Unsafe accepts crafted input: " +
                    unsafeLogin(con, "student1", attack));
            System.out.println("PreparedStatement accepts crafted input: " +
                    safeLogin(con, "student1", attack));
        }
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task05_JdbcBatch/PassLoader.java
import java.sql.Connection;
import java.sql.PreparedStatement;
import java.sql.SQLException;
import java.util.List;

public class PassLoader {
    public static int loadAll(Connection con, List<String[]> passes) throws SQLException {
        String sql = "INSERT INTO bus_pass(pass_id, reg_no, name, route_no, amount) "
                + "VALUES (?, ?, ?, ?, ?)";
        boolean oldAutoCommit = con.getAutoCommit();
        con.setAutoCommit(false);
        try (PreparedStatement ps = con.prepareStatement(sql)) {
            int count = 0;
            for (String[] pass : passes) {
                if (pass.length != 5) throw new SQLException("Pass must contain five fields");
                ps.setString(1, pass[0]);
                ps.setString(2, pass[1]);
                ps.setString(3, pass[2]);
                ps.setInt(4, Integer.parseInt(pass[3]));
                ps.setDouble(5, Double.parseDouble(pass[4]));
                ps.addBatch();
                count++;
                if (count % 1000 == 0) ps.executeBatch();
            }
            ps.executeBatch();
            con.commit();
            return count;
        } catch (SQLException | RuntimeException ex) {
            con.rollback();
            throw ex;
        } finally {
            con.setAutoCommit(oldAutoCommit);
        }
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task05_JdbcBatch/JdbcBenchmark.java
import java.sql.*;
import java.util.ArrayList;
import java.util.List;

public class JdbcBenchmark {
    private static List<String[]> makeRows(int count) {
        List<String[]> rows = new ArrayList<>();
        for (int i = 1; i <= count; i++)
            rows.add(new String[]{"P" + i, "REG" + i, "Student" + i,
                    String.valueOf((i % 20) + 1), String.valueOf(1000 + i)});
        return rows;
    }

    private static void clear(Connection con) throws SQLException {
        try (Statement st = con.createStatement()) { st.executeUpdate("DELETE FROM bus_pass"); }
    }

    public static long insertWithStatement(Connection con, List<String[]> rows)
            throws SQLException {
        clear(con);
        long start = System.nanoTime();
        try (Statement st = con.createStatement()) {
            for (String[] p : rows) {
                String sql = "INSERT INTO bus_pass(pass_id,reg_no,name,route_no,amount) VALUES ('"
                        + p[0] + "','" + p[1] + "','" + p[2] + "'," + p[3] + "," + p[4] + ")";
                st.executeUpdate(sql);
            }
        }
        return (System.nanoTime() - start) / 1_000_000;
    }

    public static long insertWithPrepared(Connection con, List<String[]> rows)
            throws SQLException {
        clear(con);
        long start = System.nanoTime();
        try (PreparedStatement ps = con.prepareStatement(
                "INSERT INTO bus_pass(pass_id,reg_no,name,route_no,amount) VALUES (?,?,?,?,?)")) {
            for (String[] p : rows) {
                bind(ps, p);
                ps.executeUpdate();
            }
        }
        return (System.nanoTime() - start) / 1_000_000;
    }

    public static long insertWithBatch(Connection con, List<String[]> rows)
            throws SQLException {
        clear(con);
        long start = System.nanoTime();
        PassLoader.loadAll(con, rows);
        return (System.nanoTime() - start) / 1_000_000;
    }

    private static void bind(PreparedStatement ps, String[] p) throws SQLException {
        ps.setString(1, p[0]); ps.setString(2, p[1]); ps.setString(3, p[2]);
        ps.setInt(4, Integer.parseInt(p[3])); ps.setDouble(5, Double.parseDouble(p[4]));
    }

    public static void main(String[] args) throws Exception {
        String url = System.getenv().getOrDefault("DB_URL",
                "jdbc:mysql://localhost:3306/assignment5");
        String user = System.getenv().getOrDefault("DB_USER", "root");
        String password = System.getenv().getOrDefault("DB_PASSWORD", "");
        try (Connection con = DriverManager.getConnection(url, user, password)) {
            for (int size : new int[]{10, 1000, 10000, 100000}) {
                List<String[]> rows = makeRows(size);
                System.out.println("ROWS=" + size);
                System.out.println("Statement ms=" + insertWithStatement(con, rows));
                System.out.println("PreparedStatement ms=" + insertWithPrepared(con, rows));
                System.out.println("Batch ms=" + insertWithBatch(con, rows));
            }
        }
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task06_Migration/PassMigration.java
import java.io.BufferedReader;
import java.nio.file.Files;
import java.nio.file.Path;
import java.sql.Connection;
import java.sql.DriverManager;
import java.sql.PreparedStatement;
import java.sql.SQLException;
import java.util.ArrayList;
import java.util.List;

public class PassMigration {
    public static void main(String[] args) throws Exception {
        String url = System.getenv().getOrDefault("DB_URL",
                "jdbc:mysql://localhost:3306/assignment5");
        String user = System.getenv().getOrDefault("DB_USER", "root");
        String password = System.getenv().getOrDefault("DB_PASSWORD", "");
        Path input = Path.of(args.length > 0 ? args[0] : "passes.csv");
        Path failures = Path.of("migration-failures.txt");
        List<String[]> valid = new ArrayList<>();

        try (BufferedReader reader = Files.newBufferedReader(input);
             var writer = Files.newBufferedWriter(failures)) {
            String line;
            int lineNo = 0;
            while ((line = reader.readLine()) != null) {
                lineNo++;
                String[] p = line.split(",", -1);
                try {
                    if (p.length != 5) throw new IllegalArgumentException("Expected 5 fields");
                    Integer.parseInt(p[3].trim());
                    double amount = Double.parseDouble(p[4].trim());
                    if (amount < 0) throw new IllegalArgumentException("Negative amount");
                    valid.add(p);
                } catch (RuntimeException ex) {
                    writer.write("Line " + lineNo + ": " + ex.getMessage() + " | " + line);
                    writer.newLine();
                }
            }
        }

        try (Connection con = DriverManager.getConnection(url, user, password)) {
            int loaded = MigrationBatchLoader.loadAll(con, valid);
            System.out.println("Valid records loaded: " + loaded);
            System.out.println("Rejected records: " + Files.readAllLines(failures).size());
            System.out.println("Failure report: " + failures.toAbsolutePath());
        }
    }
}

class MigrationBatchLoader {
    static int loadAll(Connection con, List<String[]> passes) throws SQLException {
        String sql = "INSERT INTO bus_pass(pass_id, reg_no, name, route_no, amount) VALUES (?,?,?,?,?)";
        boolean oldAutoCommit = con.getAutoCommit();
        con.setAutoCommit(false);
        try (PreparedStatement ps = con.prepareStatement(sql)) {
            int count = 0;
            for (String[] p : passes) {
                ps.setString(1, p[0]);
                ps.setString(2, p[1]);
                ps.setString(3, p[2]);
                ps.setInt(4, Integer.parseInt(p[3]));
                ps.setDouble(5, Double.parseDouble(p[4]));
                ps.addBatch();
                count++;
                if (count % 1000 == 0) ps.executeBatch();
            }
            ps.executeBatch();
            con.commit();
            return count;
        } catch (SQLException | RuntimeException ex) {
            con.rollback();
            throw ex;
        } finally {
            con.setAutoCommit(oldAutoCommit);
        }
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task06_Migration/DatabaseInspector.java
import java.sql.*;

public class DatabaseInspector {
    public static void main(String[] args) throws Exception {
        String url = System.getenv().getOrDefault("DB_URL",
                "jdbc:mysql://localhost:3306/assignment5");
        String user = System.getenv().getOrDefault("DB_USER", "root");
        String password = System.getenv().getOrDefault("DB_PASSWORD", "");
        try (Connection con = DriverManager.getConnection(url, user, password)) {
            DatabaseMetaData md = con.getMetaData();
            System.out.println("Database: " + md.getDatabaseProductName());
            System.out.println("Database version: " + md.getDatabaseProductVersion());
            System.out.println("Driver: " + md.getDriverName());
            System.out.println("Driver version: " + md.getDriverVersion());
            try (ResultSet tables = md.getTables(con.getCatalog(), null, "%",
                    new String[]{"TABLE"})) {
                while (tables.next())
                    System.out.println("Table: " + tables.getString("TABLE_NAME"));
            }
        }
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task05_JdbcBatch/MetadataAndColumnCheck.java
import java.sql.*;

public class MetadataAndColumnCheck {
    public static void inspectBusPass(Connection con) throws SQLException {
        String sql = "SELECT * FROM bus_pass";
        try (Statement st = con.createStatement(); ResultSet rs = st.executeQuery(sql)) {
            ResultSetMetaData md = rs.getMetaData();
            for (int i = 1; i <= md.getColumnCount(); i++) {
                System.out.println(md.getColumnName(i) + " | " + md.getColumnTypeName(i)
                        + " | " + md.getColumnDisplaySize(i));
            }
        }
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task01_FileStore/FileStoreExperiment.java
import java.nio.file.Files;
import java.nio.file.Path;
import java.util.Arrays;

public class FileStoreExperiment {
    public static void main(String[] args) throws Exception {
        for (int count : new int[]{1000, 10000, 50000, 100000}) {
            Path file = Files.createTempFile("pass-growth", ".csv");
            PassStore store = new PassStore(file);
            for (int i = 1; i <= count; i++)
                store.add("P" + i, "REG" + i, "Student" + i, i % 25, 1000 + i);

            long start = System.nanoTime();
            String[] found = store.find("P" + count);
            long elapsedMs = (System.nanoTime() - start) / 1_000_000;
            System.out.println("Records=" + count + ", found=" +
                    (found != null && found[0].equals("P" + count)) +
                    ", search_ms=" + elapsedMs);
            Files.deleteIfExists(file);
        }
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task01_FileStore/LineCounterBenchmark.java
import java.nio.file.Files;
import java.nio.file.Path;

public class LineCounterBenchmark {
    public static void main(String[] args) throws Exception {
        for (int sizeMb : new int[]{1, 10, 50}) {
            Path file = Files.createTempFile("attendance-" + sizeMb + "mb", ".log");
            byte[] block = new byte[1024 * 1024];
            java.util.Arrays.fill(block, (byte)'A');
            java.util.Arrays.fill(block, block.length - 1, block.length, (byte)'\n');
            try (var out = Files.newOutputStream(file)) {
                for (int i = 0; i < sizeMb; i++) out.write(block);
            }

            long start = System.nanoTime();
            long unbuffered = BufferedLineCounter.countUnbuffered(file);
            long unbufferedMs = (System.nanoTime() - start) / 1_000_000;

            start = System.nanoTime();
            long buffered = BufferedLineCounter.countBuffered(file);
            long bufferedMs = (System.nanoTime() - start) / 1_000_000;

            System.out.println("Size MB=" + sizeMb + ", lines=" + unbuffered +
                    ", unbuffered_ms=" + unbufferedMs + ", buffered_ms=" + bufferedMs);
            if (unbuffered != buffered) throw new IllegalStateException("Count mismatch");
            Files.deleteIfExists(file);
        }
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task04_Transactions/WalletTransferDemo.java
import java.math.BigDecimal;
import java.sql.*;

public class WalletTransferDemo {
    public static void main(String[] args) throws Exception {
        String url = System.getenv().getOrDefault("DB_URL",
                "jdbc:mysql://localhost:3306/assignment5");
        String user = System.getenv().getOrDefault("DB_USER", "root");
        String password = System.getenv().getOrDefault("DB_PASSWORD", "");
        try (Connection con = DriverManager.getConnection(url, user, password)) {
            WalletTransfer.transfer(con, 1, 2, new BigDecimal("500.00"));
            try (Statement st = con.createStatement();
                 ResultSet rs = st.executeQuery(
                         "SELECT wallet_id, balance FROM wallet ORDER BY wallet_id")) {
                while (rs.next())
                    System.out.println(rs.getInt(1) + " -> " + rs.getBigDecimal(2));
            }
        }
    }
}


In [ ]:
%%writefile /content/CSA09-Assignment5/Task05_JdbcBatch/schema.sql
CREATE DATABASE IF NOT EXISTS assignment5;
USE assignment5;

CREATE TABLE IF NOT EXISTS bus_pass (
    pass_id VARCHAR(20) PRIMARY KEY,
    reg_no VARCHAR(20) NOT NULL,
    name VARCHAR(100) NOT NULL,
    route_no INT NOT NULL,
    amount DECIMAL(10,2) NOT NULL
);

CREATE TABLE IF NOT EXISTS student (
    student_id INT PRIMARY KEY,
    reg_no VARCHAR(20) NOT NULL UNIQUE,
    name VARCHAR(100) NOT NULL,
    email VARCHAR(120) NOT NULL
);

CREATE TABLE IF NOT EXISTS wallet (
    wallet_id INT PRIMARY KEY,
    balance DECIMAL(12,2) NOT NULL CHECK (balance >= 0)
);

CREATE TABLE IF NOT EXISTS app_user (
    user_id VARCHAR(40) PRIMARY KEY,
    user_pwd VARCHAR(100) NOT NULL
);

INSERT INTO wallet(wallet_id, balance) VALUES
(1, 5000.00), (2, 1000.00)
ON DUPLICATE KEY UPDATE balance = VALUES(balance);

INSERT INTO app_user(user_id, user_pwd) VALUES
('student1', 'safePass123')
ON DUPLICATE KEY UPDATE user_pwd = VALUES(user_pwd);


In [ ]:
%%writefile /content/CSA09-Assignment5/Task06_Migration/passes.csv
P1001,192311001,Arun,12,4500.00
P1002,192311002,Divya,7,3800.00
P1003,192311003,Kavin,12,4200.00
BAD-LINE,ONLY-TWO
P1004,192311004,Meena,abc,3000.00
P1005,192311005,Ravi,3,2500.00


In [ ]:
TASK_FOLDERS = [
    "Task01_FileStore",
    "Task02_DataAccess",
    "Task03_Serialization",
    "Task04_Transactions",
    "Task05_JdbcBatch",
    "Task06_Migration"
]

for folder in TASK_FOLDERS:
    java_sources = sorted(str(p) for p in (PROJECT / folder).glob("*.java"))
    result = subprocess.run(
        ["javac"] + java_sources,
        cwd=PROJECT,
        capture_output=True,
        text=True
    )
    print(folder, "COMPILED" if result.returncode == 0 else "COMPILATION ERROR")
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError(f"Compilation failed in {folder}")


In [ ]:
def run_java(folder, class_name, args=None):
    command = ["java", "-cp", str(PROJECT / folder), class_name]
    if args:
        command.extend(args)
    result = subprocess.run(command, cwd=PROJECT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError(f"{class_name} failed")

run_java("Task01_FileStore", "PassStore")
run_java("Task01_FileStore", "BufferedLineCounter")
run_java("Task01_FileStore", "LineCounterBenchmark")
run_java("Task01_FileStore", "FileStoreExperiment")


In [ ]:
run_java("Task03_Serialization", "SerializationDemo")


In [ ]:
MYSQL_CONNECTOR = Path("/content/mysql-connector-j-9.2.0.jar")
if not MYSQL_CONNECTOR.exists():
    connector_url = (
        "https://repo1.maven.org/maven2/com/mysql/mysql-connector-j/"
        "9.2.0/mysql-connector-j-9.2.0.jar"
    )
    urllib.request.urlretrieve(connector_url, MYSQL_CONNECTOR)

os.environ.setdefault("DB_URL", "jdbc:mysql://YOUR_MYSQL_HOST:3306/assignment5")
os.environ.setdefault("DB_USER", "YOUR_MYSQL_USERNAME")
os.environ.setdefault("DB_PASSWORD", "YOUR_MYSQL_PASSWORD")
JDBC_CP = str(MYSQL_CONNECTOR) + ":" + ":".join(
    str(PROJECT / folder) for folder in TASK_FOLDERS
)


In [ ]:
def run_jdbc(folder, class_name, args=None):
    command = ["java", "-cp", JDBC_CP, class_name]
    if args:
        command.extend(args)
    result = subprocess.run(command, cwd=PROJECT / folder,
                            capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError(f"{class_name} failed; verify MySQL connection and schema")

# Run after the MySQL schema has been created and DB credentials have been configured.
# run_jdbc("Task04_Transactions", "LoginSecurityDemo")
# run_jdbc("Task04_Transactions", "WalletTransferDemo")
# run_jdbc("Task05_JdbcBatch", "JdbcBenchmark")
# run_jdbc("Task06_Migration", "PassMigration", ["passes.csv"])
# run_jdbc("Task06_Migration", "DatabaseInspector")


In [ ]:
ZIP_PATH = Path("/content/CSA09-Assignment5-CO5.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in PROJECT.rglob("*"):
        if path.is_file() and path.suffix != ".class":
            archive.write(path, path.relative_to(PROJECT.parent))
print(ZIP_PATH)
